# Puzzle Metadata Extraction

**Primary author:** Victoria Winters

**Builds on:**
- *DATA_RAW.md* (shared reference — §4 data architecture, §6 source-by-source extraction logic, §7 lookup table semantics)
- *data_preparation/assign_ids.py* (Victoria — produces `id_map.csv`, the `clue_id → puzzle_id` key consumed here for puzzle-level deduplication)
- *data/publisher_lookup.csv* (curated by Victoria — maps normalized extracted substrings to canonical publisher / series / setter)
- *data_preparation/puzzle_metadata.ipynb, prior version* (Victoria — earlier per-clue extraction; the normalization and lookup regexes are reused here)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Extracts puzzle-level metadata from `data/clues_raw.csv` by parsing `puzzle_name`, `source_url`, and `puzzle_date`, joins the result with `data/id_map.csv` on `clue_id`, and deduplicates to one row per `puzzle_id`. Writes `data/puzzle_metadata.csv` — a standalone shared artifact consumed by every project component. Output columns: `puzzle_id`, `blog`, `publisher`, `series`, `setter`, `puzzle_date`.


---

## §0 — Imports and paths

Standard library and pandas imports, plus a `DATA_DIR` constant and the four file paths the notebook touches. `id_map.csv` is produced by the `assign_ids.py` script rather than by any notebook — the assertion below fails fast with an actionable error message if the caller forgot to run it first.


In [ ]:
# ===
# Imports and paths
# ===
import re
import time
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data"
RAW_PATH = DATA_DIR / "clues_raw.csv"
LOOKUP_PATH = DATA_DIR / "publisher_lookup.csv"
ID_MAP_PATH = DATA_DIR / "id_map.csv"
OUTPUT_PATH = DATA_DIR / "puzzle_metadata.csv"

assert ID_MAP_PATH.exists(), (
    f"{ID_MAP_PATH} not found. "
    "Run `cd notebooks && python assign_ids.py` first."
)


---

## §1 — Load data

`clues_raw.csv` supplies the raw fields we parse: `puzzle_name`, `source_url`, and `puzzle_date`, plus `source` (used verbatim as the `blog` output column) and `clue_id` as the join key into `id_map.csv`. `keep_default_na=False` protects the crossword entry `"nan"` (grandmother) from silently becoming missing; `na_values=[""]` restores genuinely empty cells as `NaN`. `id_map.csv` is loaded for its `(clue_id, puzzle_id)` pairs and must cover the full 660,613-row raw dataset exactly once.


In [ ]:
# ===
# Load data
# ===
t0 = time.time()

df = pd.read_csv(
    RAW_PATH,
    usecols=["clue_id", "source", "puzzle_name", "source_url", "puzzle_date"],
    keep_default_na=False,
    na_values=[""],
)
print(f"clues_raw.csv           shape={df.shape}")

id_map = pd.read_csv(ID_MAP_PATH, usecols=["clue_id", "puzzle_id"])
print(f"id_map.csv              shape={id_map.shape}")

lookup_raw = pd.read_csv(LOOKUP_PATH, dtype=str).fillna("")
print(f"publisher_lookup.csv    shape={lookup_raw.shape}")

assert id_map.shape == (660_613, 2), (
    f"id_map.csv has unexpected shape {id_map.shape}; expected (660613, 2)"
)

print(f"\nLoaded all three inputs in {time.time() - t0:.1f}s")
print("\nPer-source row counts in clues_raw.csv:")
print(df["source"].value_counts().to_string())


---

## §2 — Load and prepare lookup

`publisher_lookup.csv` holds one row per `(source, field, raw)` tuple mapping the normalized extracted substring to canonical `publisher`, `series`, and `setter` values. We apply a defensive `drop_duplicates` on that composite key, restore real `NaN` values in the three output columns (they were flattened to empty strings by `fillna("")` in §1 to protect the join key), and split the table into three DataFrames — one per `field` value — so each merge in §3–§5 is a clean equi-join on `(source, raw)`.


In [ ]:
# ===
# Load and prepare lookup
# ===
lookup = (
    lookup_raw
    .drop_duplicates(subset=["source", "field", "raw"])
    .copy()
)
print(f"publisher_lookup rows after dedup: {len(lookup):,}")

# Restore real NaN values in the output columns so empty-string rows in the
# CSV don't overwrite real values during the coalescing merges in §3–§5.
for col in ["publisher", "series", "setter"]:
    lookup[col] = lookup[col].replace("", pd.NA)

lookup_leading = (
    lookup[lookup["field"] == "puzzle_name_leading"]
    [["source", "raw", "publisher", "series", "setter"]]
    .copy()
)
lookup_trailing = (
    lookup[lookup["field"] == "puzzle_name_trailing"]
    [["source", "raw", "publisher", "series", "setter"]]
    .copy()
)
lookup_url = (
    lookup[lookup["field"] == "source_url_trailing"]
    [["source", "raw", "publisher", "series", "setter"]]
    .copy()
)

print(f"\nlookup_leading   rows: {len(lookup_leading):,}")
print(f"lookup_trailing  rows: {len(lookup_trailing):,}")
print(f"lookup_url       rows: {len(lookup_url):,}")


---

## §3 — Extract `puzzle_name_leading`

Four sources encode publisher (or series) as the leading substring of `puzzle_name`, before the first run of digits: `bigdave44`, `fifteensquared`, `times_xwd_times`, and `thehinducrosswordcorner`. Normalization mirrors the lookup table's `raw` column: take the text before the first digit run, lowercase it, collapse whitespace, strip end punctuation (including en/em dashes), remove the standalone noise word `"cryptic"`, and drop a trailing standalone `"no"` acting as a puzzle-number prefix. The normalized token is joined against `lookup_leading` on `(source, raw)` to pull in `publisher` and `series`. For `times_xwd_times` the lookup supplies only the series; publisher is inferred from the series value — Sunday Times series → `"Sunday Times"`, TLS series → `"Times Literary Supplement"`, anything else → `"The Times"`.


In [ ]:
# ===
# Extract puzzle_name_leading
# ===
LEADING_SOURCES = [
    "bigdave44",
    "fifteensquared",
    "times_xwd_times",
    "thehinducrosswordcorner",
]

_END_PUNCT = r"\s\-\u2013\u2014.,:;!?'\"()\[\]"
_end_strip_re = re.compile(rf"^[{_END_PUNCT}]+|[{_END_PUNCT}]+$")
_collapse_ws_re = re.compile(r"\s+")
_pre_digit_re = re.compile(r"^(.*?)\d")

# Sloggers & Betters community-puzzle prefix detector. Raw puzzle_name
# values in fifteensquared store this series as "S B <city> ...",
# "S & B <city> ...", or "S&B <city> ..." — the ampersand is frequently
# absent in the source data, so the canonical lookup row uses the raw
# token "sloggers betters". Any leading token matching the "s b" pattern
# (with or without &) is rewritten to that canonical form so the join
# assigns publisher = "Sloggers & Betters" regardless of which city or
# setter follows. City and setter are recovered from puzzle_name in
# downstream steps, not from this leading-token lookup.
_sb_prefix_re = re.compile(r"^s\s*&?\s*b(?:\s|$)")


def normalize_leading(name):
    """Normalize the substring before the first digit run to lookup shape."""
    if not isinstance(name, str):
        return None
    m = _pre_digit_re.match(name)
    token = m.group(1) if m else name
    token = token.lower()
    token = _end_strip_re.sub("", token)
    # Strip the noise word "cryptic" as a whole token (keeps "cryptics" intact).
    token = re.sub(r"(?:^|\s)cryptic(?=\s|$)", " ", token)
    token = _collapse_ws_re.sub(" ", token).strip()
    # Strip a trailing standalone "no" acting as a puzzle-number prefix.
    token = re.sub(r"(?:^|\s)no$", "", token).strip()
    # Collapse any Sloggers & Betters abbreviation to the canonical lookup token.
    if _sb_prefix_re.match(token):
        token = "sloggers betters"
    return token


df["puzzle_name_leading_raw"] = df["puzzle_name"].apply(normalize_leading)
is_leading_source = df["source"].isin(LEADING_SOURCES)
df.loc[~is_leading_source, "puzzle_name_leading_raw"] = pd.NA

# Confirmation probe: show the normalized leading token for a sample of
# fifteensquared Sloggers & Betters puzzle_names. Expect each to collapse
# to exactly "sloggers betters" after the fix.
_sb_sample = (
    df.loc[
        (df["source"] == "fifteensquared")
        & df["puzzle_name"].str.startswith("S B", na=False),
        ["puzzle_name", "puzzle_name_leading_raw"],
    ]
    .drop_duplicates(subset=["puzzle_name"])
    .head(5)
)
print("Sloggers & Betters normalization check:")
print(_sb_sample.to_string(index=False))
print()

leading_join = lookup_leading.rename(
    columns={"raw": "puzzle_name_leading_raw"}
)
df = df.merge(
    leading_join,
    on=["source", "puzzle_name_leading_raw"],
    how="left",
    indicator="_leading_merge",
)

n_matched = int((df["_leading_merge"] == "both").sum())
print(f"puzzle_name_leading matches: {n_matched:,}")

# times_xwd_times — infer publisher from the series the lookup assigned.
times_mask = df["source"] == "times_xwd_times"
times_series = df.loc[times_mask, "series"].fillna("")

SUNDAY_SERIES = {
    "Sunday Times Cryptic", "Mephisto",
    "Christmas Special", "Christmas Extra",
}
TLS_SERIES = {"TLS Crossword"}

publisher_by_series = times_series.map(
    lambda s: "Sunday Times" if s in SUNDAY_SERIES
    else "Times Literary Supplement" if s in TLS_SERIES
    else ("The Times" if s else pd.NA)
)
df.loc[times_mask, "publisher"] = (
    df.loc[times_mask, "publisher"].fillna(publisher_by_series)
)
print(
    f"[times_xwd_times] publisher inferred — "
    f"The Times: {(publisher_by_series == 'The Times').sum():,}, "
    f"Sunday Times: {(publisher_by_series == 'Sunday Times').sum():,}, "
    f"Times Literary Supplement: "
    f"{(publisher_by_series == 'Times Literary Supplement').sum():,}"
)

df = df.drop(columns=["_leading_merge"])

---

## §4 — Extract `puzzle_name_trailing`

Two sources carry the setter in a trailing position of `puzzle_name`: `fifteensquared` (the substring after the first puzzle number, optionally after a `by` keyword) and `thehinducrosswordcorner` (the final comma-separated field after `No NNNNN, Weekday DD Mon YYYY`). Extraction follows the logic in DATA_RAW.md §6.3 and the prior notebook's §4 cells, with three additional cleanups applied before the lookup join: strip a `"with picture quiz"` suffix (a fifteensquared blog-post feature label, not setter data), strip a bare numeric suffix matching `r"\s+\d+$"` (blog disambiguation for same-day same-setter puzzles like `"Armonie 2"`), and nullify month-name bleed-through from Sloggers & Betters date-labelled entries. `"plain competition puzzle"` is deliberately left intact — the lookup table maps it directly to `setter = "Azed", series = "Plain"`. For `thehinducrosswordcorner` the series (from the §3 leading extraction) and the setter (from this step) are both preserved — `fillna` into `series` only fills cells that were still `NaN` after §3.


In [ ]:
# ===
# Extract puzzle_name_trailing
# ===
_MONTH_BLOCKLIST = {
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december",
}
_after_first_number_re = re.compile(r"\d[\d,]*\s*(.*?)\s*$")
_trailing_punct_re = re.compile(r"[\s.,;:!?'\"()\-\u2013\u2014]+$")
_leading_by_re = re.compile(r"^by\s+", re.IGNORECASE)
_picture_quiz_re = re.compile(r"\s*with picture quiz\s*$", re.IGNORECASE)
_bare_number_suffix_re = re.compile(r"\s+\d+$")


def _clean_trailing_token(token):
    """Apply post-extraction cleanups: strip blog-post noise, lowercase,
    strip trailing punctuation, and drop a blog-disambiguation numeric
    suffix like the " 2" in "Armonie 2".
    """
    token = _picture_quiz_re.sub("", token)
    token = token.lower().strip()
    token = _trailing_punct_re.sub("", token)
    token = _collapse_ws_re.sub(" ", token).strip()
    # Only strip the numeric suffix if something is left — a bare number
    # on its own should not collapse to the empty string.
    stripped = _bare_number_suffix_re.sub("", token)
    if stripped:
        token = stripped
    return token


def normalize_fifteensquared_trailing(name):
    """Extract and normalize the fifteensquared trailing token."""
    if not isinstance(name, str):
        return None
    m = _after_first_number_re.search(name)
    if not m:
        return None
    token = _leading_by_re.sub("", m.group(1)).strip()
    if not token:
        return None
    token = _clean_trailing_token(token)
    if not token or token in _MONTH_BLOCKLIST:
        return None
    return token


def normalize_hindu_trailing(name):
    """Extract and normalize the thehinducrosswordcorner trailing token.

    Sunday Crossword entries lacking a setter (fewer than 3 comma-separated
    fields) return ``None``.
    """
    if not isinstance(name, str):
        return None
    parts = [p.strip() for p in name.split(",")]
    if len(parts) < 3:
        return None
    token = _clean_trailing_token(parts[-1])
    return token or None


df["puzzle_name_trailing_raw"] = pd.NA
is_fs = df["source"] == "fifteensquared"
is_hindu = df["source"] == "thehinducrosswordcorner"
df.loc[is_fs, "puzzle_name_trailing_raw"] = (
    df.loc[is_fs, "puzzle_name"].apply(normalize_fifteensquared_trailing)
)
df.loc[is_hindu, "puzzle_name_trailing_raw"] = (
    df.loc[is_hindu, "puzzle_name"].apply(normalize_hindu_trailing)
)

trailing_join = lookup_trailing.rename(columns={
    "raw": "puzzle_name_trailing_raw",
    "publisher": "publisher_from_trailing",
    "series": "series_from_trailing",
    "setter": "setter_from_trailing",
})
df = df.merge(
    trailing_join,
    on=["source", "puzzle_name_trailing_raw"],
    how="left",
    indicator="_trailing_merge",
)

# Coalesce lookup values into the primary columns. The trailing lookup
# currently has no publisher values, but we pull the column through
# defensively so new entries don't silently lose coverage.
df["publisher"] = df["publisher"].fillna(df["publisher_from_trailing"])
df["series"] = df["series"].fillna(df["series_from_trailing"])
df["setter"] = df["setter"].fillna(df["setter_from_trailing"])

for src in ["fifteensquared", "thehinducrosswordcorner"]:
    src_mask = df["source"] == src
    extracted_mask = src_mask & df["puzzle_name_trailing_raw"].notna()
    matched_mask = extracted_mask & (df["_trailing_merge"] == "both")
    total = int(extracted_mask.sum())
    matched = int(matched_mask.sum())
    pct = (matched / total * 100) if total else 0.0
    print(f"[{src}] trailing matches: {matched:,}/{total:,} ({pct:.1f}%)")

df = df.drop(columns=[
    "publisher_from_trailing",
    "series_from_trailing",
    "setter_from_trailing",
    "_trailing_merge",
])


---

## §5 — Extract `source_url_trailing`

`thebrowser` is the only source whose setter lives in `source_url` rather than `puzzle_name`: it's the last word before the `.puz` extension, after a compressed `MonDDYY` date stamp. The lookup handles stray dashes (`"- ries"` → `"Ries"`) and collaborative puzzles (`"jacobs goodchild"` → `"Jacobs|Goodchild"`). `thebrowser` is also the one source whose raw `puzzle_date` is almost entirely missing — we reconstruct it from the parenthesized date at the end of `puzzle_name` (`"CRYPTIC #13 (March 27, 2021)"`) and emit ISO `YYYY-MM-DD` for parity with every other source.


In [ ]:
# ===
# Extract source_url_trailing
# ===
is_browser = df["source"] == "thebrowser"

_browser_setter_re = re.compile(r"[A-Z][a-z]{2}\d{4}\s+([^./]+)\.puz$")
_browser_date_re = re.compile(r"\(([^)]+)\)\s*$")


def extract_browser_url_trailing(url):
    if not isinstance(url, str):
        return None
    m = _browser_setter_re.search(url)
    if not m:
        return None
    return m.group(1).strip().lower()


def extract_browser_puzzle_date(name):
    if not isinstance(name, str):
        return None
    m = _browser_date_re.search(name)
    if not m:
        return None
    parsed = pd.to_datetime(m.group(1), errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.strftime("%Y-%m-%d")


df["source_url_trailing_raw"] = pd.NA
df.loc[is_browser, "source_url_trailing_raw"] = (
    df.loc[is_browser, "source_url"].apply(extract_browser_url_trailing)
)

url_join = lookup_url.rename(columns={
    "raw": "source_url_trailing_raw",
    "publisher": "publisher_from_url",
    "series": "series_from_url",
    "setter": "setter_from_url",
})
df = df.merge(
    url_join,
    on=["source", "source_url_trailing_raw"],
    how="left",
    indicator="_url_merge",
)
df["publisher"] = df["publisher"].fillna(df["publisher_from_url"])
df["series"] = df["series"].fillna(df["series_from_url"])
df["setter"] = df["setter"].fillna(df["setter_from_url"])

extracted_browser = is_browser & df["source_url_trailing_raw"].notna()
n_browser_matched = int(
    (extracted_browser & (df["_url_merge"] == "both")).sum()
)
n_browser_total = int(extracted_browser.sum())
pct = (n_browser_matched / n_browser_total * 100) if n_browser_total else 0.0
print(f"[thebrowser] source_url matches: "
      f"{n_browser_matched:,}/{n_browser_total:,} ({pct:.1f}%)")

df = df.drop(columns=[
    "publisher_from_url",
    "series_from_url",
    "setter_from_url",
    "_url_merge",
])

# Reconstruct puzzle_date for thebrowser where it's missing.
browser_dates = df.loc[is_browser, "puzzle_name"].apply(extract_browser_puzzle_date)
df.loc[is_browser, "puzzle_date"] = (
    df.loc[is_browser, "puzzle_date"].fillna(browser_dates)
)
n_browser_date = int(df.loc[is_browser, "puzzle_date"].notna().sum())
print(f"[thebrowser] puzzle_date populated for "
      f"{n_browser_date:,}/{int(is_browser.sum()):,} rows")


---

## §6 — Hardcoded assignments

Several sources don't use the lookup table — their publisher (and sometimes setter) are fixed by the source itself per DATA_RAW.md §6.1. `thehinducrosswordcorner` gets only its publisher hardcoded here; its series and setter come from §3 and §4 respectively. `natpostcryptic` gets both publisher and setter hardcoded (`"Hex"` is the pseudonym Emily Cox & Henry Rathvon use for the National Post). `thebrowser`'s publisher is hardcoded here; its setter was already populated from the URL lookup in §5. All assignments use `fillna` so lookup-derived values win wherever they exist.


In [ ]:
# ===
# Hardcoded assignments
# ===
HARDCODED_PUBLISHER = {
    "natpostcryptic": "National Post",
    "cru_cryptics": "Cru Cryptics Forum",
    "nytimes": "New York Times",
    "leoedit": "Amuselabs Leoedit",
    "newyorker": "The New Yorker",
    "thehinducrosswordcorner": "The Hindu",
    "thebrowser": "The Browser",
}
HARDCODED_SETTER = {
    "natpostcryptic": "Hex",
}

for src, pub in HARDCODED_PUBLISHER.items():
    mask = df["source"] == src
    df.loc[mask, "publisher"] = df.loc[mask, "publisher"].fillna(pub)
    print(f"[{src}] publisher := {pub!r}  ({int(mask.sum()):,} rows)")

for src, setter in HARDCODED_SETTER.items():
    mask = df["source"] == src
    df.loc[mask, "setter"] = df.loc[mask, "setter"].fillna(setter)
    print(f"[{src}] setter := {setter!r}  ({int(mask.sum()):,} rows)")


---

## §7 — Assemble clue-level frame

Project the working DataFrame down to the six puzzle-level columns plus `clue_id`, rename `source` to `blog`, and join against `id_map.csv` on `clue_id` to attach `puzzle_id`. The assertions verify that the merge is lossless and that every raw-data row received exactly one non-null `puzzle_id`.


In [ ]:
# ===
# Assemble clue-level frame
# ===
clue_level = df[[
    "clue_id", "source", "publisher", "series", "setter", "puzzle_date",
]].rename(columns={"source": "blog"})

clue_level = clue_level.merge(id_map, on="clue_id", how="left")

assert len(clue_level) == 660_613, (
    f"row count changed during id_map merge: {len(clue_level):,}"
)
assert clue_level["puzzle_id"].notna().all(), (
    "some clue_ids are missing from id_map.csv"
)

print(f"Clue-level frame shape: {clue_level.shape}")
print(f"Distinct puzzle_ids:    {clue_level['puzzle_id'].nunique():,}")


---

## §8 — Deduplicate to puzzle level

Group by `puzzle_id` and take the first value of each puzzle-level column. Every clue belonging to the same puzzle should already carry identical values for `blog`, `publisher`, `series`, `setter`, and `puzzle_date` (because those values were derived from puzzle-level raw fields), so `"first"` is sufficient — sorting by `clue_id` first gives deterministic output across reruns. The result must be exactly 22,628 rows, matching the puzzle count `assign_ids.py` produces.


In [ ]:
# ===
# Deduplicate to puzzle level
# ===
puzzle_level = (
    clue_level
    .sort_values("clue_id", kind="stable")
    .groupby("puzzle_id", as_index=False)
    .agg({
        "blog": "first",
        "publisher": "first",
        "series": "first",
        "setter": "first",
        "puzzle_date": "first",
    })
)

assert len(puzzle_level) == 22_628, (
    f"Unexpected puzzle count: {len(puzzle_level):,}; expected 22,628"
)

puzzle_level["puzzle_id"] = puzzle_level["puzzle_id"].astype(int)
puzzle_level = puzzle_level[[
    "puzzle_id", "blog", "publisher", "series", "setter", "puzzle_date",
]]

print(f"Puzzle-level frame shape: {puzzle_level.shape}")
print()
print("Per-blog coverage (non-NaN %):")
coverage_cols = ["publisher", "series", "setter", "puzzle_date"]
coverage = (
    puzzle_level
    .groupby("blog")[coverage_cols]
    .apply(lambda g: g.notna().mean())
)
with pd.option_context("display.float_format", "{:.1%}".format):
    print(coverage.to_string())


---

## §9 — Unmatched token log

For each of the three extraction fields, list every `(source, raw)` combination that did not match any entry in `publisher_lookup.csv`, grouped by source with row counts. These tables drive follow-up work on the lookup table — legitimate setter/series names that should be added, noise patterns that should be filtered more aggressively, and malformed rows that should be flagged unresolvable. A markdown cell beneath each table is left for interpretation notes after the notebook runs.


In [ ]:
# ===
# Unmatched tokens — puzzle_name_leading
# ===
def log_unmatched(field_name, raw_col, sources, lookup_df):
    """Print all (source, raw) combinations with no match in ``lookup_df``,
    grouped by source, with row counts.
    """
    keys = set(zip(lookup_df["source"], lookup_df["raw"]))
    for src in sources:
        sub = df[(df["source"] == src) & df[raw_col].notna()]
        unmatched = sub[~sub.apply(
            lambda r: (r["source"], r[raw_col]) in keys, axis=1
        )]
        counts = unmatched[raw_col].value_counts()
        print(f"── {src} / {field_name} ── "
              f"{len(counts):,} unique unmatched tokens, "
              f"{int(len(unmatched)):,} rows ──")
        if counts.empty:
            print("  (none)")
        else:
            for tok, cnt in counts.items():
                print(f"  {cnt:>6,}  {tok!r}")
        print()


log_unmatched(
    "puzzle_name_leading",
    "puzzle_name_leading_raw",
    LEADING_SOURCES,
    lookup_leading,
)


### Analysis — unmatched `puzzle_name_leading` tokens

*[TO FILL IN after running]* — Describe which sources carry the largest unmatched counts, whether each frequent token looks like noise (`__trashed`, single letters, month names, blank strings, title leakage) or a legitimate series name that should be added to `publisher_lookup.csv`, and what (if anything) to queue as a follow-up CSV edit. `thehinducrosswordcorner / puzzle_name_leading` is expected to carry a large empty-string bucket corresponding to the main Hindu series — this is already absorbed by the §6 hardcoded publisher assignment and is not a genuine gap.


In [ ]:
# ===
# Unmatched tokens — puzzle_name_trailing
# ===
log_unmatched(
    "puzzle_name_trailing",
    "puzzle_name_trailing_raw",
    ["fifteensquared", "thehinducrosswordcorner"],
    lookup_trailing,
)


### Analysis — unmatched `puzzle_name_trailing` tokens

*[TO FILL IN after running]* — Note unmatched fifteensquared tokens that look like legitimate setter pseudonyms (candidates to add to `publisher_lookup.csv`) versus noise tokens like stray publisher names, month-name bleed not covered by the block list, or puzzle-title leakage. For `thehinducrosswordcorner`, flag any remaining setter variants or punctuation oddities beyond the `Dr. X` / `Spinner` / `KrisKross` normalizations already in the lookup.


In [ ]:
# ===
# Unmatched tokens — source_url_trailing
# ===
log_unmatched(
    "source_url_trailing",
    "source_url_trailing_raw",
    ["thebrowser"],
    lookup_url,
)


### Analysis — unmatched `source_url_trailing` tokens

*[TO FILL IN after running]* — Note any `thebrowser` surnames not currently in `publisher_lookup.csv`. Given the small size of this source (84 puzzles), the list should be short; each row is a candidate for manual verification against the Browser's published setter list.


---

## §10 — Write output

Overwrite `data/puzzle_metadata.csv` with the deduplicated puzzle-level frame. Per CLAUDE.md this is a shared upstream artifact — do not rerun this notebook without team agreement and a documented reason in `DECISIONS.md`.


In [ ]:
# ===
# Write output
# ===
puzzle_level.to_csv(OUTPUT_PATH, index=False)
size_kb = OUTPUT_PATH.stat().st_size / 1024
print(f"Wrote {len(puzzle_level):,} rows to {OUTPUT_PATH} ({size_kb:.1f} KB)")
print()
print(puzzle_level.head().to_string(index=False))


---

## §11 — Summary

### Inputs

- `data/clues_raw.csv` — 660,613 rows (read: `clue_id`, `source`, `puzzle_name`, `source_url`, `puzzle_date`).
- `data/id_map.csv` — 660,613 rows, produced by `data_preparation/assign_ids.py` (read: `clue_id`, `puzzle_id`).
- `data/publisher_lookup.csv` — curated mapping from normalized extracted substrings to canonical `publisher`, `series`, and `setter`.

### Output

`data/puzzle_metadata.csv` — 22,628 rows, one per `puzzle_id`. Columns in order:

| column | type | notes |
|---|---|---|
| `puzzle_id` | int | Unique key for grouping clues at the puzzle grain; produced by `assign_ids.py`. |
| `blog` | str | The raw `source` value from `clues_raw.csv`. Acts as a coarse provenance tag. |
| `publisher` | str | Canonical publisher name; `NaN` where not available. |
| `series` | str | Puzzle series within the publication; `NaN` where not applicable or unknown. |
| `setter` | str | Pipe-separated for collaborative puzzles (e.g. `"Enigmatist|Soup"`); `NaN` where unknown. |
| `puzzle_date` | str | Publication date. Passed through from `clues_raw.csv` for every source except `thebrowser`, where it is reconstructed from `puzzle_name` and emitted as ISO `YYYY-MM-DD`. |

### Join pattern for downstream use

`clues_filtered.csv` is keyed on `clue_id`, which is **not** unique after double-definition expansion. To attach puzzle metadata, join through `id_map.csv`:

```python
clues = pd.read_csv("data/clues_filtered.csv")
id_map = pd.read_csv("data/id_map.csv")
meta = pd.read_csv("data/puzzle_metadata.csv")
joined = clues.merge(id_map, on="clue_id").merge(meta, on="puzzle_id")
```

Puzzle-level columns will repeat across every clue belonging to the same puzzle — this is correct and expected.

### Known limitations

- `publisher`, `series`, and `setter` are only as complete as `publisher_lookup.csv`. Unmatched tokens are logged in §9 so that legitimate gaps can be filled in the lookup CSV.
- `setter` is stored as a pipe-separated string, never as a Python list. Downstream code that needs individual setter names should call `.str.split("|")`.
- `puzzle_date` for most sources is passed through verbatim from the raw data and is **not** normalized to a single format here. Downstream components that need a uniform date representation should parse it at read time.
- The grain of this file is per-puzzle. Clue-level attributes (`clue_no`, `clue_direction`, `puzzle_no`) belong in a future `clue_metadata.csv`, not here.
- `times_xwd_times` setter coverage is 0% by design — Times crossword setters are anonymous and not recorded in the source data.